# Week 2 — Exploratory Data Analysis and Visualization

## Exploratory Data Analysis of India's Monetary and Banking Indicators

**Dataset:** RBI Monetary / Banking Indicators Dataset

### Objective
This notebook performs a complete professional EDA:
- Dataset profiling and data quality checks
- Date validation and cleaning
- Feature engineering
- Descriptive statistics
- Univariate, bivariate and multivariate analysis
- Time-series analysis
- Correlation analysis
- Outlier and anomaly detection
- Monthly, quarterly and yearly analysis
- Report-ready tables and visualizations

> **Report standard:** Every major visualization should be followed in the final DOC by an **Observation**, **Interpretation**, and **Critical Discussion**.


In [ ]:

# =============================================================================
# WEEK 2 TASK
# EXPLORATORY DATA ANALYSIS AND VISUALIZATION
# =============================================================================
# Project Title:
# Exploratory Data Analysis of India's Monetary and Banking Indicators
#
# Dataset:
# RBI Monetary / Banking Indicators Dataset
#
# Objective:
# Perform a complete, reproducible, professional EDA using Pandas, Matplotlib,
# and Seaborn. The analysis covers data understanding, data quality, cleaning,
# feature engineering, descriptive statistics, univariate/bivariate/
# multivariate analysis, time-series patterns, correlation, outliers,
# anomaly detection, and report-ready insights.
#
# NOTE:
# This script is designed for the uploaded RBI CSV. Put the CSV in the same
# folder as this script or change FILE_PATH below.
# =============================================================================

# =============================================================================
# 0. IMPORTS AND CONFIGURATION
# =============================================================================

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# Professional plotting defaults
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.titlesize"] = 15
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["legend.fontsize"] = 10

# ----------------------------- FILE SETTINGS -------------------------------
FILE_PATH = "df309dad-9533-4a3a-9ee1-41dffe93634a.csv"

OUTPUT_DIR = Path("RBI_Week2_Professional_Outputs")
PLOT_DIR = OUTPUT_DIR / "01_visualizations"
TABLE_DIR = OUTPUT_DIR / "02_tables"
REPORT_DIR = OUTPUT_DIR / "03_report_material"
DATA_DIR = OUTPUT_DIR / "04_processed_data"

for folder in [OUTPUT_DIR, PLOT_DIR, TABLE_DIR, REPORT_DIR, DATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


# =============================================================================
# 1. HELPER FUNCTIONS
# =============================================================================

def save_plot(filename):
    """Save the current Matplotlib figure as a high-resolution PNG."""
    plt.tight_layout()
    plt.savefig(PLOT_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


def section(title):
    """Print a clean section heading."""
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def first_existing(columns, candidates):
    """Return the first candidate that exists in columns."""
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None


def iqr_outliers(data, column):
    """Return outliers and IQR boundaries for a numerical column."""
    s = data[column].dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (data[column] < lower) | (data[column] > upper)
    return data.loc[mask].copy(), q1, q3, iqr, lower, upper


def add_value_labels(ax, fmt="{:.1f}"):
    """Add labels to bars."""
    for container in ax.containers:
        try:
            ax.bar_label(container, fmt=fmt, padding=3, fontsize=8)
        except Exception:
            pass


# =============================================================================
# 2. LOAD DATA
# =============================================================================

section("1. DATA LOADING")

df = pd.read_csv(FILE_PATH)

print(f"Dataset loaded successfully.")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]:,}")
print(f"Memory  : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

df.head().to_csv(REPORT_DIR / "sample_first_5_rows.csv", index=False)


# =============================================================================
# 3. DATASET PROFILE
# =============================================================================

section("2. DATASET PROFILE")

profile = pd.DataFrame({
    "column": df.columns,
    "data_type": df.dtypes.astype(str).values,
    "non_null": df.notna().sum().values,
    "missing": df.isna().sum().values,
    "missing_pct": (df.isna().mean().values * 100).round(2),
    "unique_values": df.nunique(dropna=True).values
})

print(profile.to_string(index=False))
profile.to_csv(TABLE_DIR / "01_dataset_profile.csv", index=False)

print("\nShape:", df.shape)
print("\nDuplicate rows:", df.duplicated().sum())

duplicate_count = int(df.duplicated().sum())

# Save duplicate rows if any
if duplicate_count > 0:
    df.loc[df.duplicated(keep=False)].to_csv(
        TABLE_DIR / "02_duplicate_rows.csv", index=False
    )


# =============================================================================
# 4. DATA TYPE AND DATE VALIDATION
# =============================================================================

section("3. DATE AND DATA-TYPE VALIDATION")

if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.sort_values("Date").reset_index(drop=True)

    print("Date range:")
    print("Start:", df["Date"].min())
    print("End  :", df["Date"].max())
    print("Invalid dates:", df["Date"].isna().sum())

# Ensure known numeric columns are numeric
for col in df.columns:
    if col not in ["Country", "Month", "month_name", "Date"]:
        # Do not force object columns blindly; convert only if mostly numeric.
        converted = pd.to_numeric(df[col], errors="coerce")
        if converted.notna().mean() >= 0.90:
            df[col] = converted

print("\nData types after validation:")
print(df.dtypes.to_string())


# =============================================================================
# 5. IDENTIFY IMPORTANT COLUMNS
# =============================================================================

section("4. IMPORTANT COLUMN MAPPING")

# Exact names from the uploaded dataset
M3_RAW = "M3 (UOM:INR(IndianRupees)), Scaling Factor:10000000"
M3_MERGER_RAW = "M3 Including Merger (UOM:INR(IndianRupees)), Scaling Factor:10000000"
CURRENCY_RAW = "Currency With The Public (UOM:INR(IndianRupees)), Scaling Factor:10000000"
DEMAND_RAW = "Demand Deposits With Banks (UOM:INR(IndianRupees)), Scaling Factor:10000000"
TIME_RAW = "Time Deposits With Banks (UOM:INR(IndianRupees)), Scaling Factor:10000000"
OTHER_RBI_RAW = "Other Deposits With Reserve Bank (UOM:INR(IndianRupees)), Scaling Factor:10000000"
GOVT_CREDIT_RAW = "Sources Of Net Bank Credit To Government (UOM:INR(IndianRupees)), Scaling Factor:10000000"
COMMERCIAL_RAW = "Sources Of Bank Credit To Commercial Sector (UOM:INR(IndianRupees)), Scaling Factor:10000000"
FX_RAW = "Sources Of Net Foreign Exchange Assets Of Banking Sector (UOM:INR(IndianRupees)), Scaling Factor:10000000"
GOVT_LIAB_RAW = "Sources Of Governments Currency Liabilities To The Public (UOM:INR(IndianRupees)), Scaling Factor:10000000"
NON_MONETARY_RAW = "Sources Of Banking Sectors Net Non Monetary Liabilities (UOM:INR(IndianRupees)), Scaling Factor:10000000"

rename_map = {
    M3_RAW: "M3",
    M3_MERGER_RAW: "M3_including_merger",
    CURRENCY_RAW: "currency_public",
    DEMAND_RAW: "demand_deposits",
    TIME_RAW: "time_deposits",
    OTHER_RBI_RAW: "other_rbi_deposits",
    GOVT_CREDIT_RAW: "net_bank_credit_government",
    COMMERCIAL_RAW: "commercial_credit",
    FX_RAW: "net_fx_assets",
    GOVT_LIAB_RAW: "government_currency_liabilities",
    NON_MONETARY_RAW: "net_non_monetary_liabilities",
}

rename_map = {k: v for k, v in rename_map.items() if k in df.columns}
df = df.rename(columns=rename_map)

print("Renamed analytical columns:")
for old, new in rename_map.items():
    print(f"  {new:<35} <- {old}")

# Required analytical variables
core_vars = [
    "M3",
    "M3_including_merger",
    "currency_public",
    "demand_deposits",
    "time_deposits",
    "other_rbi_deposits",
    "net_bank_credit_government",
    "commercial_credit",
    "net_fx_assets",
    "government_currency_liabilities",
    "net_non_monetary_liabilities",
]

core_vars = [c for c in core_vars if c in df.columns]

if "M3" not in df.columns:
    raise ValueError(
        "M3 column was not found. Please check the uploaded CSV column name."
    )

print("\nCore variables available for analysis:")
print(core_vars)


# =============================================================================
# 6. MISSING VALUE ANALYSIS
# =============================================================================

section("5. MISSING VALUE ANALYSIS")

missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(3)
}).sort_values("missing_count", ascending=False)

print(missing.to_string())
missing.to_csv(TABLE_DIR / "03_missing_value_analysis.csv")

# Visualize missingness only if there are missing values
missing_positive = missing[missing["missing_count"] > 0]

if not missing_positive.empty:
    plt.figure(figsize=(11, 5))
    ax = sns.barplot(
        x=missing_positive.index,
        y=missing_positive["missing_percentage"]
    )
    plt.title("Missing Values by Column")
    plt.xlabel("Column")
    plt.ylabel("Missing Values (%)")
    plt.xticks(rotation=75, ha="right")
    add_value_labels(ax, fmt="{:.2f}")
    save_plot("01_missing_values.png")
else:
    print("No missing values detected.")


# =============================================================================
# 7. HANDLE EXPECTED FIRST-ROW PERCENTAGE-CHANGE MISSING VALUES
# =============================================================================

section("6. EXPECTED MISSING VALUES IN GROWTH VARIABLES")

# Existing engineered variables in the uploaded file
existing_m3_change = "m3_pct_change" if "m3_pct_change" in df.columns else None
existing_fx_change = (
    "net_fx_assets_pct_change"
    if "net_fx_assets_pct_change" in df.columns
    else None
)

# Recalculate from source values for reproducibility
df["m3_pct_change_calc"] = df["M3"].pct_change() * 100

if "net_fx_assets" in df.columns:
    df["net_fx_assets_pct_change_calc"] = (
        df["net_fx_assets"].pct_change() * 100
    )

print("M3 growth calculation created from raw M3.")
if "net_fx_assets" in df.columns:
    print("FX growth calculation created from raw net FX assets.")

print("\nFirst observations:")
growth_cols = ["Date", "M3", "m3_pct_change_calc"]
if "net_fx_assets" in df.columns:
    growth_cols += ["net_fx_assets", "net_fx_assets_pct_change_calc"]
print(df[growth_cols].head())


# =============================================================================
# 8. DUPLICATE HANDLING
# =============================================================================

section("7. DUPLICATE HANDLING")

before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)

print("Rows before duplicate removal:", before)
print("Rows after duplicate removal :", after)
print("Duplicates removed            :", before - after)


# =============================================================================
# 9. FEATURE ENGINEERING
# =============================================================================

section("8. FEATURE ENGINEERING")

if "Date" in df.columns:
    df["analysis_year"] = df["Date"].dt.year
    df["analysis_month"] = df["Date"].dt.month
    df["analysis_month_name"] = df["Date"].dt.month_name()
    df["analysis_quarter"] = "Q" + df["Date"].dt.quarter.astype(str)
    df["analysis_day"] = df["Date"].dt.day
    df["analysis_week"] = df["Date"].dt.isocalendar().week.astype(int)

# Deposit aggregate
deposit_cols = [
    c for c in ["demand_deposits", "time_deposits", "other_rbi_deposits"]
    if c in df.columns
]
if deposit_cols:
    df["total_selected_deposits"] = df[deposit_cols].sum(axis=1)

# Credit aggregate where available
credit_cols = [
    c for c in ["net_bank_credit_government", "commercial_credit"]
    if c in df.columns
]
if credit_cols:
    df["total_selected_credit"] = df[credit_cols].sum(axis=1)

# Year-over-year growth for M3 if enough time observations exist
if "M3" in df.columns and "analysis_year" in df.columns:
    yearly_m3 = df.groupby("analysis_year")["M3"].mean()
    yearly_growth = yearly_m3.pct_change() * 100
    df["annual_average_m3_growth_pct"] = df["analysis_year"].map(yearly_growth)

print("Engineered variables:")
for c in [
    "analysis_year",
    "analysis_month",
    "analysis_month_name",
    "analysis_quarter",
    "analysis_day",
    "analysis_week",
    "total_selected_deposits",
    "total_selected_credit",
    "annual_average_m3_growth_pct",
]:
    if c in df.columns:
        print(" -", c)


# =============================================================================
# 10. DESCRIPTIVE STATISTICS
# =============================================================================

section("9. DESCRIPTIVE STATISTICS")

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

descriptive = df[numeric_cols].describe().T

descriptive["range"] = descriptive["max"] - descriptive["min"]
descriptive["coefficient_of_variation_pct"] = (
    descriptive["std"] / descriptive["mean"].replace(0, np.nan)
) * 100

descriptive = descriptive.round(4)

print(descriptive.to_string())

descriptive.to_csv(
    TABLE_DIR / "04_descriptive_statistics_all_numeric.csv"
)

core_summary = (
    df[core_vars]
    .describe()
    .T
    .round(4)
)

core_summary.to_csv(
    TABLE_DIR / "05_core_indicator_statistics.csv"
)


# =============================================================================
# 11. DATASET PERIOD SUMMARY
# =============================================================================

section("10. TIME PERIOD SUMMARY")

if "Date" in df.columns:
    period_summary = pd.DataFrame({
        "metric": [
            "first_date",
            "last_date",
            "number_of_monthly_records",
            "number_of_years",
        ],
        "value": [
            df["Date"].min(),
            df["Date"].max(),
            len(df),
            df["Date"].dt.year.nunique(),
        ],
    })

    print(period_summary.to_string(index=False))
    period_summary.to_csv(
        REPORT_DIR / "time_period_summary.csv", index=False
    )


# =============================================================================
# 12. UNIVARIATE ANALYSIS — M3
# =============================================================================

section("11. UNIVARIATE ANALYSIS — M3")

plt.figure(figsize=(12, 6))
sns.histplot(df["M3"].dropna(), bins=30, kde=True)
plt.title("Distribution of M3")
plt.xlabel("M3")
plt.ylabel("Frequency")
save_plot("02_m3_distribution.png")


# =============================================================================
# 13. UNIVARIATE ANALYSIS — M3 GROWTH
# =============================================================================

plt.figure(figsize=(12, 6))
sns.histplot(df["m3_pct_change_calc"].dropna(), bins=30, kde=True)
plt.title("Distribution of Monthly M3 Growth")
plt.xlabel("Monthly M3 Growth (%)")
plt.ylabel("Frequency")
save_plot("03_m3_growth_distribution.png")


# =============================================================================
# 14. TIME SERIES — M3
# =============================================================================

section("12. TIME-SERIES ANALYSIS — M3")

plt.figure(figsize=(14, 6))
plt.plot(
    df["Date"],
    df["M3"],
    linewidth=2,
    label="M3"
)
plt.title("M3 Trend Over Time")
plt.xlabel("Date")
plt.ylabel("M3")
plt.legend()
plt.xticks(rotation=45)
save_plot("04_m3_trend.png")


# =============================================================================
# 15. TIME SERIES — M3 GROWTH
# =============================================================================

plt.figure(figsize=(14, 6))
plt.plot(
    df["Date"],
    df["m3_pct_change_calc"],
    linewidth=1.5,
    label="Monthly M3 Growth"
)
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Monthly Percentage Change in M3")
plt.xlabel("Date")
plt.ylabel("M3 Growth (%)")
plt.legend()
plt.xticks(rotation=45)
save_plot("05_m3_monthly_growth.png")


# =============================================================================
# 16. TIME SERIES — CURRENCY
# =============================================================================

if "currency_public" in df.columns:
    plt.figure(figsize=(14, 6))
    plt.plot(
        df["Date"],
        df["currency_public"],
        linewidth=2
    )
    plt.title("Currency With the Public Over Time")
    plt.xlabel("Date")
    plt.ylabel("Currency With the Public")
    plt.xticks(rotation=45)
    save_plot("06_currency_with_public.png")


# =============================================================================
# 17. TIME SERIES — DEPOSITS
# =============================================================================

if all(c in df.columns for c in ["demand_deposits", "time_deposits"]):
    plt.figure(figsize=(14, 6))
    plt.plot(
        df["Date"],
        df["demand_deposits"],
        label="Demand Deposits",
        linewidth=2
    )
    plt.plot(
        df["Date"],
        df["time_deposits"],
        label="Time Deposits",
        linewidth=2
    )
    plt.title("Demand Deposits vs Time Deposits")
    plt.xlabel("Date")
    plt.ylabel("Deposit Value")
    plt.legend()
    plt.xticks(rotation=45)
    save_plot("07_demand_vs_time_deposits.png")


# =============================================================================
# 18. TIME SERIES — COMMERCIAL CREDIT
# =============================================================================

if "commercial_credit" in df.columns:
    plt.figure(figsize=(14, 6))
    plt.plot(
        df["Date"],
        df["commercial_credit"],
        linewidth=2
    )
    plt.title("Bank Credit to Commercial Sector Over Time")
    plt.xlabel("Date")
    plt.ylabel("Commercial Credit")
    plt.xticks(rotation=45)
    save_plot("08_commercial_credit.png")


# =============================================================================
# 19. TIME SERIES — NET FOREIGN EXCHANGE ASSETS
# =============================================================================

if "net_fx_assets" in df.columns:
    plt.figure(figsize=(14, 6))
    plt.plot(
        df["Date"],
        df["net_fx_assets"],
        linewidth=2
    )
    plt.title("Net Foreign Exchange Assets Over Time")
    plt.xlabel("Date")
    plt.ylabel("Net Foreign Exchange Assets")
    plt.xticks(rotation=45)
    save_plot("09_net_fx_assets.png")


# =============================================================================
# 20. M3 VS M3 INCLUDING MERGER
# =============================================================================

if "M3_including_merger" in df.columns:
    plt.figure(figsize=(14, 6))
    plt.plot(df["Date"], df["M3"], label="M3", linewidth=2)
    plt.plot(
        df["Date"],
        df["M3_including_merger"],
        label="M3 Including Merger",
        linewidth=2,
        linestyle="--"
    )
    plt.title("M3 vs M3 Including Merger")
    plt.xlabel("Date")
    plt.ylabel("Value")
    plt.legend()
    plt.xticks(rotation=45)
    save_plot("10_m3_vs_m3_merger.png")


# =============================================================================
# 21. BOXPLOTS — KEY VARIABLES
# =============================================================================

section("13. OUTLIER SCREENING")

box_vars = [
    c for c in [
        "M3",
        "currency_public",
        "demand_deposits",
        "time_deposits",
        "commercial_credit",
        "net_fx_assets",
        "m3_pct_change_calc",
    ]
    if c in df.columns
]

for variable in box_vars:
    plt.figure(figsize=(11, 4.5))
    sns.boxplot(x=df[variable])
    plt.title(f"Boxplot: {variable}")
    plt.xlabel(variable)
    save_plot(f"boxplot_{variable}.png")


# =============================================================================
# 22. IQR OUTLIER REPORT
# =============================================================================

outlier_rows = []

for variable in box_vars:
    temp = df[[c for c in ["Date", variable] if c in df.columns]].copy()
    outliers, q1, q3, iqr, lower, upper = iqr_outliers(df, variable)

    outlier_rows.append({
        "variable": variable,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": len(outliers),
        "outlier_percentage": len(outliers) / len(df) * 100,
    })

    if len(outliers) > 0:
        keep = [c for c in ["Date", variable] if c in outliers.columns]
        outliers[keep].to_csv(
            TABLE_DIR / f"outliers_{variable}.csv",
            index=False
        )

outlier_summary = pd.DataFrame(outlier_rows).round(4)
outlier_summary.to_csv(
    TABLE_DIR / "06_iqr_outlier_summary.csv",
    index=False
)

print(outlier_summary.to_string(index=False))


# =============================================================================
# 23. CORRELATION MATRIX
# =============================================================================

section("14. CORRELATION ANALYSIS")

corr_vars = [
    c for c in [
        "M3",
        "currency_public",
        "demand_deposits",
        "time_deposits",
        "other_rbi_deposits",
        "net_bank_credit_government",
        "commercial_credit",
        "net_fx_assets",
        "government_currency_liabilities",
        "net_non_monetary_liabilities",
    ]
    if c in df.columns
]

corr_matrix = df[corr_vars].corr()

corr_matrix.to_csv(
    TABLE_DIR / "07_correlation_matrix.csv"
)

plt.figure(figsize=(12, 9))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5
)
plt.title("Correlation Matrix of Major Monetary and Banking Indicators")
save_plot("11_correlation_heatmap.png")


# =============================================================================
# 24. M3 CORRELATION RANKING
# =============================================================================

if "M3" in corr_matrix.columns:
    m3_corr = (
        corr_matrix["M3"]
        .drop("M3")
        .sort_values(key=lambda x: x.abs(), ascending=False)
        .to_frame("correlation_with_M3")
    )

    m3_corr.to_csv(
        TABLE_DIR / "08_m3_correlation_ranking.csv"
    )

    print("\nVariables ranked by absolute correlation with M3:")
    print(m3_corr)


# =============================================================================
# 25. BIVARIATE ANALYSIS — M3 VS COMMERCIAL CREDIT
# =============================================================================

if "commercial_credit" in df.columns:
    plt.figure(figsize=(9, 6))
    sns.regplot(
        data=df,
        x="commercial_credit",
        y="M3",
        scatter_kws={"alpha": 0.55},
        line_kws={"linewidth": 2}
    )
    plt.title("Relationship Between M3 and Commercial Credit")
    plt.xlabel("Commercial Credit")
    plt.ylabel("M3")
    save_plot("12_m3_vs_commercial_credit_regression.png")


# =============================================================================
# 26. BIVARIATE ANALYSIS — M3 VS TIME DEPOSITS
# =============================================================================

if "time_deposits" in df.columns:
    plt.figure(figsize=(9, 6))
    sns.regplot(
        data=df,
        x="time_deposits",
        y="M3",
        scatter_kws={"alpha": 0.55},
        line_kws={"linewidth": 2}
    )
    plt.title("Relationship Between M3 and Time Deposits")
    plt.xlabel("Time Deposits")
    plt.ylabel("M3")
    save_plot("13_m3_vs_time_deposits_regression.png")


# =============================================================================
# 27. BIVARIATE ANALYSIS — M3 VS DEMAND DEPOSITS
# =============================================================================

if "demand_deposits" in df.columns:
    plt.figure(figsize=(9, 6))
    sns.regplot(
        data=df,
        x="demand_deposits",
        y="M3",
        scatter_kws={"alpha": 0.55},
        line_kws={"linewidth": 2}
    )
    plt.title("Relationship Between M3 and Demand Deposits")
    plt.xlabel("Demand Deposits")
    plt.ylabel("M3")
    save_plot("14_m3_vs_demand_deposits_regression.png")


# =============================================================================
# 28. MONTHLY SEASONAL ANALYSIS
# =============================================================================

section("15. MONTHLY AND QUARTERLY PATTERNS")

if "analysis_month_name" in df.columns:
    month_order = [
        "January", "February", "March", "April",
        "May", "June", "July", "August",
        "September", "October", "November", "December"
    ]

    monthly_summary = (
        df.groupby("analysis_month_name")
        .agg(
            average_M3=("M3", "mean"),
            median_M3=("M3", "median"),
            average_M3_growth=("m3_pct_change_calc", "mean"),
        )
        .reindex(month_order)
    )

    monthly_summary.to_csv(
        TABLE_DIR / "09_monthly_summary.csv"
    )

    plt.figure(figsize=(13, 6))
    ax = sns.barplot(
        x=monthly_summary.index,
        y=monthly_summary["average_M3"]
    )
    plt.title("Average M3 by Calendar Month")
    plt.xlabel("Month")
    plt.ylabel("Average M3")
    plt.xticks(rotation=45)
    save_plot("15_average_m3_by_month.png")


# =============================================================================
# 29. QUARTERLY ANALYSIS
# =============================================================================

if "analysis_quarter" in df.columns:
    quarterly_summary = (
        df.groupby("analysis_quarter")
        .agg(
            average_M3=("M3", "mean"),
            median_M3=("M3", "median"),
            average_M3_growth=("m3_pct_change_calc", "mean"),
        )
        .reindex(["Q1", "Q2", "Q3", "Q4"])
    )

    quarterly_summary.to_csv(
        TABLE_DIR / "10_quarterly_summary.csv"
    )

    plt.figure(figsize=(9, 6))
    sns.barplot(
        x=quarterly_summary.index,
        y=quarterly_summary["average_M3"]
    )
    plt.title("Average M3 by Quarter")
    plt.xlabel("Quarter")
    plt.ylabel("Average M3")
    save_plot("16_average_m3_by_quarter.png")


# =============================================================================
# 30. YEARLY SUMMARY
# =============================================================================

if "analysis_year" in df.columns:
    yearly_agg = {
        "average_M3": ("M3", "mean"),
        "median_M3": ("M3", "median"),
        "M3_growth_average_pct": ("m3_pct_change_calc", "mean"),
    }

    optional_yearly = {
        "average_currency_public": "currency_public",
        "average_demand_deposits": "demand_deposits",
        "average_time_deposits": "time_deposits",
        "average_commercial_credit": "commercial_credit",
        "average_net_fx_assets": "net_fx_assets",
    }

    for output_name, source_name in optional_yearly.items():
        if source_name in df.columns:
            yearly_agg[output_name] = (source_name, "mean")

    yearly_summary = df.groupby("analysis_year").agg(**yearly_agg)
    yearly_summary.to_csv(TABLE_DIR / "11_yearly_summary.csv")

    print("\nYearly summary:")
    print(yearly_summary.round(3).to_string())


# =============================================================================
# 31. YEARLY M3 TREND
# =============================================================================

if "analysis_year" in df.columns:
    plt.figure(figsize=(13, 6))
    plt.plot(
        yearly_summary.index,
        yearly_summary["average_M3"],
        marker="o",
        linewidth=2
    )
    plt.title("Average M3 by Year")
    plt.xlabel("Year")
    plt.ylabel("Average M3")
    plt.grid(True, alpha=0.3)
    save_plot("17_yearly_average_m3.png")


# =============================================================================
# 32. YEARLY COMMERCIAL CREDIT
# =============================================================================

if (
    "analysis_year" in df.columns
    and "average_commercial_credit" in yearly_summary.columns
):
    plt.figure(figsize=(13, 6))
    plt.plot(
        yearly_summary.index,
        yearly_summary["average_commercial_credit"],
        marker="o",
        linewidth=2
    )
    plt.title("Average Commercial Credit by Year")
    plt.xlabel("Year")
    plt.ylabel("Average Commercial Credit")
    plt.grid(True, alpha=0.3)
    save_plot("18_yearly_commercial_credit.png")


# =============================================================================
# 33. TOP AND BOTTOM M3 GROWTH PERIODS
# =============================================================================

section("16. ANOMALY AND EXTREME-MOVEMENT ANALYSIS")

growth_data = df.dropna(subset=["m3_pct_change_calc"]).copy()

top_growth = (
    growth_data.nlargest(10, "m3_pct_change_calc")
    [["Date", "M3", "m3_pct_change_calc"]]
)

bottom_growth = (
    growth_data.nsmallest(10, "m3_pct_change_calc")
    [["Date", "M3", "m3_pct_change_calc"]]
)

top_growth.to_csv(
    TABLE_DIR / "12_top_10_m3_growth_periods.csv",
    index=False
)

bottom_growth.to_csv(
    TABLE_DIR / "13_bottom_10_m3_growth_periods.csv",
    index=False
)

print("\nTop 10 M3 growth periods:")
print(top_growth.to_string(index=False))

print("\nBottom 10 M3 growth periods:")
print(bottom_growth.to_string(index=False))


# =============================================================================
# 34. EXTREME FX MOVEMENTS
# =============================================================================

if "net_fx_assets_pct_change_calc" in df.columns:
    fx_growth_data = df.dropna(
        subset=["net_fx_assets_pct_change_calc"]
    )

    top_fx = (
        fx_growth_data.nlargest(
            10,
            "net_fx_assets_pct_change_calc"
        )
        [["Date", "net_fx_assets", "net_fx_assets_pct_change_calc"]]
    )

    bottom_fx = (
        fx_growth_data.nsmallest(
            10,
            "net_fx_assets_pct_change_calc"
        )
        [["Date", "net_fx_assets", "net_fx_assets_pct_change_calc"]]
    )

    top_fx.to_csv(
        TABLE_DIR / "14_top_10_fx_growth_periods.csv",
        index=False
    )

    bottom_fx.to_csv(
        TABLE_DIR / "15_bottom_10_fx_growth_periods.csv",
        index=False
    )


# =============================================================================
# 35. M3 COMPOSITION / COMPONENT COMPARISON
# =============================================================================

component_vars = [
    c for c in [
        "currency_public",
        "demand_deposits",
        "time_deposits",
        "other_rbi_deposits",
    ]
    if c in df.columns
]

if component_vars:
    component_means = (
        df[component_vars]
        .mean()
        .sort_values(ascending=False)
    )

    component_means.to_csv(
        TABLE_DIR / "16_average_monetary_components.csv"
    )

    plt.figure(figsize=(11, 6))
    sns.barplot(
        x=component_means.values,
        y=component_means.index
    )
    plt.title("Average Value of Selected Monetary Components")
    plt.xlabel("Average Value")
    plt.ylabel("Monetary Component")
    save_plot("19_average_monetary_components.png")


# =============================================================================
# 36. M3 VS TOTAL SELECTED DEPOSITS
# =============================================================================

if "total_selected_deposits" in df.columns:
    plt.figure(figsize=(9, 6))
    sns.regplot(
        data=df,
        x="total_selected_deposits",
        y="M3",
        scatter_kws={"alpha": 0.55},
        line_kws={"linewidth": 2}
    )
    plt.title("M3 vs Total Selected Deposits")
    plt.xlabel("Total Selected Deposits")
    plt.ylabel("M3")
    save_plot("20_m3_vs_total_deposits.png")


# =============================================================================
# 37. ROLLING TREND / SMOOTHING
# =============================================================================

section("17. ROLLING-TREND ANALYSIS")

df["M3_3_month_moving_average"] = (
    df["M3"].rolling(window=3, min_periods=1).mean()
)

plt.figure(figsize=(14, 6))
plt.plot(
    df["Date"],
    df["M3"],
    label="Monthly M3",
    alpha=0.5
)
plt.plot(
    df["Date"],
    df["M3_3_month_moving_average"],
    label="3-Month Moving Average",
    linewidth=2.5
)
plt.title("M3 with 3-Month Moving Average")
plt.xlabel("Date")
plt.ylabel("M3")
plt.legend()
plt.xticks(rotation=45)
save_plot("21_m3_moving_average.png")


# =============================================================================
# 38. PERIOD-TO-PERIOD CHANGE
# =============================================================================

df["M3_absolute_change"] = df["M3"].diff()

change_summary = df[
    ["Date", "M3", "M3_absolute_change", "m3_pct_change_calc"]
].copy()

change_summary.to_csv(
    TABLE_DIR / "17_m3_period_changes.csv",
    index=False
)


# =============================================================================
# 39. CORRELATION OF GROWTH RATES
# =============================================================================

growth_columns = [
    c for c in [
        "m3_pct_change_calc",
        "net_fx_assets_pct_change_calc"
    ]
    if c in df.columns
]

if len(growth_columns) >= 2:
    growth_corr = df[growth_columns].corr()

    growth_corr.to_csv(
        TABLE_DIR / "18_growth_rate_correlation.csv"
    )

    plt.figure(figsize=(7, 5))
    sns.heatmap(
        growth_corr,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0
    )
    plt.title("Correlation of Key Growth Rates")
    save_plot("22_growth_rate_correlation.png")


# =============================================================================
# 40. AUTOMATIC KEY-FINDING TABLE
# =============================================================================

section("18. AUTOMATED KEY-FINDING SUMMARY")

key_findings = []

# M3 trend
if len(df) >= 2:
    first_m3 = df["M3"].iloc[0]
    last_m3 = df["M3"].iloc[-1]
    overall_change = ((last_m3 / first_m3) - 1) * 100

    key_findings.append({
        "category": "M3 overall trend",
        "finding": f"M3 changed by approximately {overall_change:.2f}% "
                    f"between the first and last observations."
    })

# M3 growth extremes
if not growth_data.empty:
    max_growth = growth_data.loc[
        growth_data["m3_pct_change_calc"].idxmax()
    ]
    min_growth = growth_data.loc[
        growth_data["m3_pct_change_calc"].idxmin()
    ]

    key_findings.append({
        "category": "Highest M3 growth",
        "finding": f"{max_growth['Date'].date()} recorded the highest "
                    f"monthly M3 growth of "
                    f"{max_growth['m3_pct_change_calc']:.2f}%."
    })

    key_findings.append({
        "category": "Lowest M3 growth",
        "finding": f"{min_growth['Date'].date()} recorded the lowest "
                    f"monthly M3 growth of "
                    f"{min_growth['m3_pct_change_calc']:.2f}%."
    })

# Strongest M3 correlation
if "M3" in corr_matrix.columns and len(corr_matrix.columns) > 1:
    corr_without_self = corr_matrix["M3"].drop("M3")
    strongest_variable = corr_without_self.abs().idxmax()
    strongest_value = corr_without_self.loc[strongest_variable]

    key_findings.append({
        "category": "Strongest association with M3",
        "finding": f"{strongest_variable} has the strongest absolute "
                    f"correlation with M3 ({strongest_value:.3f})."
    })

# Outlier count
m3_outlier_info = outlier_summary[
    outlier_summary["variable"] == "M3"
]

if not m3_outlier_info.empty:
    count = int(m3_outlier_info["outlier_count"].iloc[0])
    pct = float(m3_outlier_info["outlier_percentage"].iloc[0])

    key_findings.append({
        "category": "M3 outliers",
        "finding": f"IQR screening identified {count} M3 observations "
                    f"({pct:.2f}% of records) as statistical outliers."
    })

key_findings_df = pd.DataFrame(key_findings)

print(key_findings_df.to_string(index=False))

key_findings_df.to_csv(
    REPORT_DIR / "automated_key_findings.csv",
    index=False
)


# =============================================================================
# 41. FINAL DATA QUALITY REPORT
# =============================================================================

section("19. FINAL DATA QUALITY REPORT")

final_quality = pd.DataFrame({
    "metric": [
        "rows",
        "columns",
        "duplicate_rows",
        "missing_cells",
        "date_min",
        "date_max",
    ],
    "value": [
        len(df),
        df.shape[1],
        df.duplicated().sum(),
        int(df.isna().sum().sum()),
        df["Date"].min() if "Date" in df.columns else None,
        df["Date"].max() if "Date" in df.columns else None,
    ]
})

print(final_quality.to_string(index=False))

final_quality.to_csv(
    REPORT_DIR / "final_data_quality_report.csv",
    index=False
)


# =============================================================================
# 42. SAVE PROCESSED DATASET
# =============================================================================

section("20. SAVE FINAL PROCESSED DATA")

final_dataset_path = DATA_DIR / "RBI_final_processed_dataset.csv"

df.to_csv(
    final_dataset_path,
    index=False
)

print("Final processed dataset saved to:")
print(final_dataset_path)


# =============================================================================
# 43. CREATE A REPORT-READY INDEX OF VISUALIZATIONS
# =============================================================================

plot_files = sorted(
    [p.name for p in PLOT_DIR.glob("*.png")]
)

plot_index = pd.DataFrame({
    "visualization_number": range(1, len(plot_files) + 1),
    "file_name": plot_files,
})

plot_index.to_csv(
    REPORT_DIR / "visualization_index.csv",
    index=False
)

print("\nVisualization files created:", len(plot_files))


# =============================================================================
# 44. FINAL COMPLETION MESSAGE
# =============================================================================

section("EDA PROJECT COMPLETED")

print("Project output folder:")
print(OUTPUT_DIR.resolve())

print("\nFolders:")
print("1. Visualizations :", PLOT_DIR.resolve())
print("2. Tables         :", TABLE_DIR.resolve())
print("3. Report Material:", REPORT_DIR.resolve())
print("4. Processed Data :", DATA_DIR.resolve())

print("\nThe analysis includes:")
print("✓ Dataset profiling")
print("✓ Data-type validation")
print("✓ Missing-value analysis")
print("✓ Duplicate checking")
print("✓ Date processing")
print("✓ Feature engineering")
print("✓ Descriptive statistics")
print("✓ Univariate analysis")
print("✓ Time-series analysis")
print("✓ Bivariate analysis")
print("✓ Multivariate correlation analysis")
print("✓ Distribution analysis")
print("✓ IQR outlier analysis")
print("✓ Extreme-movement analysis")
print("✓ Monthly analysis")
print("✓ Quarterly analysis")
print("✓ Yearly analysis")
print("✓ Moving-average analysis")
print("✓ Automated key findings")
print("✓ Report-ready tables")
print("✓ High-resolution visualizations")

print("\nIMPORTANT:")
print("Use the generated tables and PNG files in the final DOC report.")
print("Do not paste every graph into the report; select the strongest")
print("visualizations and write an observation + interpretation below each.")
